In [1]:
# Installs to get OPUS-MT
!pip install transformers sacrebleu sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.3 MB/s eta 0:00:00


In [2]:
# Mount drive to retrieve datasets
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
# Load Europarl test set — same split as IBM Model 1
fr_path = '/content/drive/MyDrive/cs505_data/europarl-v7.fr-en.fr'
en_path = '/content/drive/MyDrive/cs505_data/europarl-v7.fr-en.en'

with open(fr_path, encoding='utf-8') as f:
    fr_lines = f.readlines()
with open(en_path, encoding='utf-8') as f:
    en_lines = f.readlines()

TRAIN_SIZE = 50000
TEST_SIZE  = 2000

test_fr = [s.strip() for s in fr_lines[TRAIN_SIZE:TRAIN_SIZE + TEST_SIZE]]
test_en = [s.strip() for s in en_lines[TRAIN_SIZE:TRAIN_SIZE + TEST_SIZE]]

print(f"Test sentences: {len(test_fr):,}")
print(f"Sample FR: {test_fr[0]}")
print(f"Sample EN: {test_en[0]}")

Test sentences: 2,000
Sample FR: Le nombre de cas augmente. C'est bien la tendance qui pose un problème ici, plus encore que l'importance de l'épidémie sur le plan quantitatif.
Sample EN: It is this tendency which presents a problem right now, much more than the scale of the epidemic in quantitative terms.


In [4]:
from transformers import MarianMTModel, MarianTokenizer

model_name = "Helsinki-NLP/opus-mt-fr-en"
model = MarianMTModel.from_pretrained(model_name)
tokenizer = MarianTokenizer.from_pretrained(model_name)
print("Model loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

...

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

Model loaded successfully


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [8]:
from tqdm import tqdm
import sacrebleu

# BLEU Eval
def translate_opus(sentences, batch_size=32):
    translations = []
    for i in tqdm(range(0, len(sentences), batch_size)):
        batch = sentences[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512)
        translated = model.generate(**inputs)
        decoded = [tokenizer.decode(t, skip_special_tokens=True) for t in translated]
        translations.extend(decoded)
    return translations

translations_opus = translate_opus(test_fr)

bleu = sacrebleu.corpus_bleu(translations_opus, [test_en])
print(f"\nOPUS-MT Transformer BLEU score: {bleu.score:.2f}")

100%|██████████| 63/63 [46:06<00:00, 43.92s/it]



OPUS-MT Transformer BLEU score: 32.53


OPUS-MT Transformer BLEU score: 37.99

BLEU Score=32.53 on Europarl Test Split

Leaving here if outputs don't save;
Training took more than an hour.

In [9]:
# Tests

print("\nSample translations:")
for i in range(5):
    print(f"\nFR:  {test_fr[i]}")
    print(f"REF: {test_en[i]}")
    print(f"OPUS: {translations_opus[i]}")




Sample translations:

FR:  Le nombre de cas augmente. C'est bien la tendance qui pose un problème ici, plus encore que l'importance de l'épidémie sur le plan quantitatif.
REF: It is this tendency which presents a problem right now, much more than the scale of the epidemic in quantitative terms.
OPUS: The number of cases is increasing, and that is the problem here, even more so than the importance of the epidemic in terms of quantity.

FR:  En effet, à l'heure actuelle, on ne sait tout simplement pas l'expliquer, d'où l'ouverture d'un débat sur l'existence possible de voies de transmission de la maladie que nous ignorerions encore.
REF: The truth is, at the present time, there is quite simply no adequate explanation for it, hence the debate which has been initiated on the possible existence of alternative disease transmission routes of which we are, as yet, unaware.
OPUS: Indeed, at the moment, we simply cannot explain it, hence the opening of a debate on the possible existence of rout